<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN found:", HF_TOKEN is not None)

HF_TOKEN found: True


In [ ]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connected successfully.")
print("March 2026 data path is ready.")

DuckDB connected successfully.
March 2026 data path is ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract

1. **Unit of analysis:**  
One row in my final feature frame represents one pseudonymized content page within one pseudonymized client.

2. **Table:**  
I will use the `fact_content_daily_performance` table.

3. **Time window:**  
I will use the March 2026 partition. The feature window is March 1–15, 2026, and the later proxy-outcome window is March 16–31, 2026.

4. **Prediction or ranking target:**  
I will score and rank pages by their risk of a meaningful decline in average daily search impressions during the later half of the month. The provisional proxy equals 1 when the second-half daily average is more than 20% lower than the first-half daily average.

5. **Deliberate exclusion:**  
I will exclude second-half impressions and every column calculated from them from the honest feature set because they are not knowable at the decision moment.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature fields

1. **imp_first_half** — total GSC impressions observed from March 1–15.
2. **clicks_first_half** — total GSC clicks observed from March 1–15.
3. **ctr_first_half** — clicks divided by impressions during March 1–15.
4. **avg_position_first_half** — average GSC search position during March 1–15.
5. **active_days_first_half** — number of days with at least one impression during March 1–15.

### Label / proxy

**is_declining_proxy** — equals 1 when average daily impressions during March 16–31 are less than 80% of average daily impressions during March 1–15; otherwise it equals 0.
### Context fields

**client_hash_id** and **content_hash_id** are used for grouping, identification, and train/test splitting. They are not predictive features.

### Excluded fields

**imp_second_half** and any column calculated from second-half impressions are excluded from the honest feature set because they use information from the outcome window and would cause data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Verification Query 1 — Grain check

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {FACT_MARCH}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate grain groups:", len(grain_check))
display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain groups: 0


,report_date,client_hash_id,content_hash_id,row_count


The grain check returned zero duplicate groups. This supports the raw-table grain of one report date × one pseudonymized client × one pseudonymized content page. The final feature frame will later aggregate these daily rows into one row per client-content page.

In [ ]:
# Verification Query 2 — Row count and date span

count_and_dates = con.sql(f"""
    SELECT
        COUNT(*) AS raw_row_count,
        COUNT(
            DISTINCT client_hash_id || '|' || content_hash_id
        ) AS distinct_client_content_pages,
        MIN(report_date) AS minimum_date,
        MAX(report_date) AS maximum_date
    FROM {FACT_MARCH}
""").df()

display(count_and_dates)

,raw_row_count,distinct_client_content_pages,minimum_date,maximum_date
0,9841378,331437,2026-03-01,2026-03-31


The March 2026 slice contains 9,841,378 raw daily rows and 331,437 distinct client-content pages. The observed date span is March 1, 2026 to March 31, 2026. This confirms that the selected partition covers the full mid-panel month used for this task.

In [ ]:
# Verification Query 3 — GSC availability

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows_after_gsc_filter,

        COUNT(
            DISTINCT client_hash_id || '|' || content_hash_id
        ) AS pages_after_gsc_filter

    FROM {FACT_MARCH}

    WHERE gsc_data_available IS TRUE
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_after_gsc_filter,pages_after_gsc_filter
0,3611061,176738


After filtering with `gsc_data_available IS TRUE`, 3,611,061 daily rows and 176,738 distinct client-content pages remain. These rows have available Google Search Console measurements and are suitable for building the search-based features in this slice.

In [ ]:
# Build the five-feature frame

feature_source = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS imp_first_half,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS clicks_first_half,

        100.0 * SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_clicks
                ELSE 0
            END
        ) / NULLIF(
            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS ctr_first_half,

        AVG(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                     AND gsc_avg_position > 0
                THEN gsc_avg_position
            END
        ) AS avg_position_first_half,

        COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                     AND gsc_impressions > 0
                THEN report_date
            END
        ) AS active_days_first_half,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_impressions
                ELSE 0
            END
        ) / NULLIF(
            COUNT(
                DISTINCT CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                    THEN report_date
                END
            ),
            0
        ) AS avg_daily_imp_first_half,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-16'
                                     AND DATE '2026-03-31'
                THEN gsc_impressions
                ELSE 0
            END
        ) / NULLIF(
            COUNT(
                DISTINCT CASE
                    WHEN report_date BETWEEN DATE '2026-03-16'
                                         AND DATE '2026-03-31'
                    THEN report_date
                END
            ),
            0
        ) AS avg_daily_imp_second_half

    FROM {FACT_MARCH}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING
        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_impressions
                ELSE 0
            END
        ) >= 100

        AND COUNT(
            DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-03-16'
                                     AND DATE '2026-03-31'
                THEN report_date
            END
        ) > 0
""").df()


feature_source["is_declining_proxy"] = (
    feature_source["avg_daily_imp_second_half"]
    < 0.80 * feature_source["avg_daily_imp_first_half"]
).astype(int)


FEATURES = [
    "imp_first_half",
    "clicks_first_half",
    "ctr_first_half",
    "avg_position_first_half",
    "active_days_first_half",
]


feature_frame = feature_source[
    [
        "client_hash_id",
        "content_hash_id",
        *FEATURES,
        "is_declining_proxy",
    ]
].copy()


print("Feature-frame rows:", len(feature_frame))
print("Feature count:", len(FEATURES))
print(
    "Declining-proxy rate:",
    round(feature_frame["is_declining_proxy"].mean(), 3)
)

display(feature_frame.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 77400
Feature count: 5
Declining-proxy rate: 0.345


,client_hash_id,content_hash_id,imp_first_half,clicks_first_half,ctr_first_half,avg_position_first_half,active_days_first_half,is_declining_proxy
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,0.143781,6.327311,15,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,0.000000,4.185913,15,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,0.080972,6.473735,15,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,0.327869,7.259861,15,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,0.416667,3.860842,15,1
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,131.0,0.0,0.000000,9.284735,15,1
6,client_73cda7b4e4f265ea,content_22c063002b7c1caf,172.0,0.0,0.000000,7.602850,15,1
7,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,3104.0,16.0,0.515464,5.536679,15,0
8,client_73cda7b4e4f265ea,content_20403327d8d9374c,1294.0,5.0,0.386399,6.902681,15,0
9,client_73cda7b4e4f265ea,content_f8df6b20d18c4374,527.0,1.0,0.189753,7.522099,15,1


### Feature availability

- **imp_first_half** is knowable at the decision moment because it uses only GSC impressions observed from March 1–15.
- **clicks_first_half** is knowable at the decision moment because it uses only GSC clicks observed from March 1–15.
- **ctr_first_half** is knowable at the decision moment because it is calculated only from first-half clicks and impressions.
- **avg_position_first_half** is knowable at the decision moment because it uses only search-position measurements observed before March 16.
- **active_days_first_half** is knowable at the decision moment because it counts only first-half days with at least one impression.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score


model_data = feature_frame.dropna(
    subset=["is_declining_proxy"]
).reset_index(drop=True)


X = model_data[FEATURES]
y = model_data["is_declining_proxy"]
groups = model_data["client_hash_id"]


splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_index, test_index = next(
    splitter.split(X, y, groups=groups)
)


honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(
        max_depth=3,
        random_state=42,
    ),
)


honest_model.fit(
    X.iloc[train_index],
    y.iloc[train_index],
)


honest_predictions = honest_model.predict(
    X.iloc[test_index]
)


honest_score = balanced_accuracy_score(
    y.iloc[test_index],
    honest_predictions,
)


print(
    "Test declining-proxy rate:",
    round(y.iloc[test_index].mean(), 3)
)

print(
    "Honest balanced accuracy:",
    round(honest_score, 3)
)

Test declining-proxy rate: 0.233
Honest balanced accuracy: 0.496


In [ ]:
# Deliberate leakage experiment

leak_frame = model_data.copy()

# This column is an intentional copy of the label.
# It would never be allowed in an honest model.
leak_frame["label_copy_leak"] = (
    leak_frame["is_declining_proxy"]
)


LEAKED_FEATURES = FEATURES + ["label_copy_leak"]

X_leaked = leak_frame[LEAKED_FEATURES]


leaked_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(
        max_depth=3,
        random_state=42,
    ),
)


leaked_model.fit(
    X_leaked.iloc[train_index],
    y.iloc[train_index],
)


leaked_predictions = leaked_model.predict(
    X_leaked.iloc[test_index]
)


leaked_score = balanced_accuracy_score(
    y.iloc[test_index],
    leaked_predictions,
)


print(
    "Honest balanced accuracy:",
    round(honest_score, 3)
)

print(
    "Leaked balanced accuracy:",
    round(leaked_score, 3)
)

Honest balanced accuracy: 0.496
Leaked balanced accuracy: 1.0


In [ ]:
leak_frame = leak_frame.drop(
    columns=["label_copy_leak"]
)

assert "label_copy_leak" not in leak_frame.columns
assert len(FEATURES) == 5

print("Leak column removed.")
print("Final honest features:", FEATURES)
print("Final feature count:", len(FEATURES))
print(
    "Score kept for reporting:",
    round(honest_score, 3)
)

Leak column removed.
Final honest features: ['imp_first_half', 'clicks_first_half', 'ctr_first_half', 'avg_position_first_half', 'active_days_first_half']
Final feature count: 5
Score kept for reporting: 0.496


### Leakage lesson

The honest five-feature model achieved a balanced accuracy of **0.496**.

After I deliberately added `label_copy_leak`, which was copied directly from the proxy label, the balanced accuracy increased to **1.000**. This near-perfect result was artificial because the answer was included among the model inputs.

I deleted the leaked column and kept the honest five-feature frame and the honest score.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named limitation — short proxy window

This slice compares the first and second halves of one month. A measured short-term decline may reflect temporary volatility, reporting patterns, or seasonality rather than a durable content-refresh opportunity.

The proxy can support directional ranking and human review, but it cannot prove that refreshing a recommended page will cause traffic recovery.

## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x ] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.